In [ ]:
import yaml
from rdflib import Graph, Namespace, RDF, RDFS, URIRef, Literal

# script to convert ontology.yaml to ontology.ttl
# this is needed to properly load it into fuseki
# and then query it with SPARQL


with open("ontology.yaml", "r", encoding="utf-8") as f:
    ontology = yaml.safe_load(f)

EX = Namespace(ontology.get("namespace", "http://example.org/food#"))
g = Graph()
g.bind(ontology.get("prefix", "ex"), EX)

for cls, info in ontology.get("classes", {}).items():
    uri = EX[cls]
    g.add((uri, RDF.type, RDFS.Class))
    if "description" in info:
        g.add((uri, RDFS.comment, Literal(info["description"])))

for rel, info in ontology.get("relations", {}).items():
    uri = EX[rel]
    g.add((uri, RDF.type, RDF.Property))
    if "domain" in info:
        g.add((uri, RDFS.domain, EX[info["domain"]]))
    if "range" in info:
        g.add((uri, RDFS.range, EX[info["range"]]))
    if "description" in info:
        g.add((uri, RDFS.comment, Literal(info["description"])))

g.serialize("ontology.ttl", format="turtle")
print("ontology.ttl generated successfully!")


ontology.ttl generated successfully!


In [7]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# connect to local Fuseki dataset
fuseki_url = "http://localhost:3030/nutrition_pdf/query"
sparql = SPARQLWrapper(fuseki_url)
sparql.setReturnFormat(JSON)

# function to run queries and display results
def run_query(query, title):
    sparql.setQuery(query)
    results = sparql.query().convert()
    df = pd.DataFrame([
        {k: b[k]["value"] for k in b}
        for b in results["results"]["bindings"]
    ])
    print(f"\n{title}")
    print(df.head(10).to_string(index=False))
    return df


# count all triples
query1 = """
SELECT (COUNT(*) AS ?totalTriples)
WHERE { ?s ?p ?o . }
"""
run_query(query1, "Total number of triples in the graph")



Total number of triples in the graph
totalTriples
        1789


,totalTriples
0,1789


In [8]:

# list ingredients and their nutrients
query2 = """
PREFIX ex: <http://example.org/food#>
SELECT ?ingredient ?nutrient
WHERE {
  ?ingredient ex:hasNutrient ?nutrient .
}
LIMIT 20
"""
run_query(query2, "Ingredients and their nutrients")


Ingredients and their nutrients
                       ingredient                        nutrient
  http://example.org/food#alcohol http://example.org/food#calcium
  http://example.org/food#berries http://example.org/food#calcium
http://example.org/food#beverages http://example.org/food#calcium
  http://example.org/food#cereals http://example.org/food#calcium
    http://example.org/food#dairy http://example.org/food#calcium
     http://example.org/food#fats http://example.org/food#calcium
     http://example.org/food#fish http://example.org/food#calcium
   http://example.org/food#fruits http://example.org/food#calcium
   http://example.org/food#grains http://example.org/food#calcium
  http://example.org/food#legumes http://example.org/food#calcium


,ingredient,nutrient
0,http://example.org/food#alcohol,http://example.org/food#calcium
1,http://example.org/food#berries,http://example.org/food#calcium
2,http://example.org/food#beverages,http://example.org/food#calcium
3,http://example.org/food#cereals,http://example.org/food#calcium
4,http://example.org/food#dairy,http://example.org/food#calcium
5,http://example.org/food#fats,http://example.org/food#calcium
6,http://example.org/food#fish,http://example.org/food#calcium
7,http://example.org/food#fruits,http://example.org/food#calcium
8,http://example.org/food#grains,http://example.org/food#calcium
9,http://example.org/food#legumes,http://example.org/food#calcium


In [10]:

# find ingredients linked to environmental impacts
query3 = """
PREFIX ex: <http://example.org/food#>
SELECT ?ingredient ?impact
WHERE {
  ?ingredient ex:hasEnvironmentalImpact ?impact .
}
ORDER BY ?ingredient
LIMIT 20
"""
run_query(query3, "Ingredients with environmental impacts")


Ingredients with environmental impacts
                     ingredient                                       impact
http://example.org/food#alcohol http://example.org/food#environmental_impact
http://example.org/food#alcohol       http://example.org/food#sustainability
 http://example.org/food#barley         http://example.org/food#biodiversity
 http://example.org/food#barley       http://example.org/food#climate_impact
 http://example.org/food#barley http://example.org/food#environmental_impact
 http://example.org/food#barley      http://example.org/food#water_footprint
http://example.org/food#berries         http://example.org/food#biodiversity
http://example.org/food#berries     http://example.org/food#carbon_footprint
http://example.org/food#berries       http://example.org/food#climate_impact
http://example.org/food#berries http://example.org/food#environmental_impact


,ingredient,impact
0,http://example.org/food#alcohol,http://example.org/food#environmental_impact
1,http://example.org/food#alcohol,http://example.org/food#sustainability
2,http://example.org/food#barley,http://example.org/food#biodiversity
3,http://example.org/food#barley,http://example.org/food#climate_impact
4,http://example.org/food#barley,http://example.org/food#environmental_impact
5,http://example.org/food#barley,http://example.org/food#water_footprint
6,http://example.org/food#berries,http://example.org/food#biodiversity
7,http://example.org/food#berries,http://example.org/food#carbon_footprint
8,http://example.org/food#berries,http://example.org/food#climate_impact
9,http://example.org/food#berries,http://example.org/food#environmental_impact


In [12]:

# discover nutrients related to health outcomes
query4 = """
PREFIX ex: <http://example.org/food#>
SELECT ?nutrient ?outcome
WHERE {
  ?nutrient ex:affectsRiskOf ?outcome .
}
ORDER BY ?nutrient
LIMIT 20
"""
run_query(query4, "Nutrients and associated health outcomes")


Nutrients and associated health outcomes
                       nutrient                                   outcome
http://example.org/food#calcium    http://example.org/food#blood_pressure
http://example.org/food#calcium http://example.org/food#colorectal_cancer
http://example.org/food#calcium   http://example.org/food#ldl_cholesterol
http://example.org/food#calcium         http://example.org/food#mortality
http://example.org/food#calcium            http://example.org/food#stroke
  http://example.org/food#fibre    http://example.org/food#blood_pressure
  http://example.org/food#fibre http://example.org/food#colorectal_cancer
  http://example.org/food#fibre      http://example.org/food#hypertension
  http://example.org/food#fibre   http://example.org/food#ldl_cholesterol
  http://example.org/food#fibre         http://example.org/food#mortality


,nutrient,outcome
0,http://example.org/food#calcium,http://example.org/food#blood_pressure
1,http://example.org/food#calcium,http://example.org/food#colorectal_cancer
2,http://example.org/food#calcium,http://example.org/food#ldl_cholesterol
3,http://example.org/food#calcium,http://example.org/food#mortality
4,http://example.org/food#calcium,http://example.org/food#stroke
5,http://example.org/food#fibre,http://example.org/food#blood_pressure
6,http://example.org/food#fibre,http://example.org/food#colorectal_cancer
7,http://example.org/food#fibre,http://example.org/food#hypertension
8,http://example.org/food#fibre,http://example.org/food#ldl_cholesterol
9,http://example.org/food#fibre,http://example.org/food#mortality


In [16]:
query5 = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?entity
       (xsd:integer(STRBEFORE(STRAFTER(STR(?comment), "Found on page "), ",")) AS ?page)
WHERE {
  ?entity rdfs:comment ?comment .
  FILTER(CONTAINS(STR(?comment), "Found on page"))
}
ORDER BY ?page
LIMIT 20
"""
run_query(query5, "Entities and their source PDF pages")



Entities and their source PDF pages
                                entity page
       http://example.org/food#berries    5
          http://example.org/food#fish    5
        http://example.org/food#fruits    5
        http://example.org/food#grains    5
  http://example.org/food#increase_the    5
       http://example.org/food#legumes    5
     http://example.org/food#limit_the    5
          http://example.org/food#nuts    5
          http://example.org/food#oils    5
http://example.org/food#prefer_sources    5


,entity,page
0,http://example.org/food#berries,5
1,http://example.org/food#fish,5
2,http://example.org/food#fruits,5
3,http://example.org/food#grains,5
4,http://example.org/food#increase_the,5
5,http://example.org/food#legumes,5
6,http://example.org/food#limit_the,5
7,http://example.org/food#nuts,5
8,http://example.org/food#oils,5
9,http://example.org/food#prefer_sources,5
